## 2.4 TodoListMiddleware中间件

TodoListMiddleware中间件赋予了Agent 任务规划 和 追踪进度 的能力，可以 应对复杂的多步任务 。

比如，当一个大任务需要被拆解为 3 个以上的子任务，且前面的步骤是后面步骤的前提时，如果不列Todo 列表，大模型在执行到第 3 步时，很容易忘记自己最初的目标，或者在工具返回大量报错信息后“应激”，直接跳过验证去回答用户。

此时，TodoListMiddleware 中间件会给模型一个 `write_todos` 工具，并在每次调用模型前附上使用说明，引导它先列计划、再按计划执行，每完成一步就更新任务状态（用不用由模型自己决定）。每次写入的计划都会留在消息历史里，模型据此随时知道“下一步该干什么”；计划同时保存在 state 的 `todos` 字段中，方便程序读取（详见 2.4.3）。

你的任务是否需要拆解？

    ├── 否 (比如：问答、翻译、单次函数调用) ──> ❌ 绝不需要，浪费算力

    └── 是 (比如：写一个包含多文件的工程)

        └── 步骤是否多变且需要应对失败？

            ├── 否 (步骤完全固定，如 A->B->C) ──> ❌ 传统的LangGraph线性节点即可
            
            └── 是 (AI 需要边做边调计划) ──> 引入 TodoListMiddleware

典型场景：
* 任务链路长、步骤多，且有严格的先后依赖关系
* 需要在前端 UI 界面实时展示 Agent 的“思考与执行进度”
* ....

To-do list的创建和维护是通过调用 `write_todos`工具 实现的。

> 如果把普通的 Agent 比作“想到哪写到哪”的实习生，那么引入了 TodoListMiddleware 的 Agent就是“先写方案、再列 CheckList、最后按部就班执行”的资深工程师。

```python
TodoListMiddleware(
  self,
  *,
  system_prompt: str = WRITE_TODOS_SYSTEM_PROMPT,
  tool_description: str = WRITE_TODOS_TOOL_DESCRIPTION
)
```

### 2.4.1 参数说明
1. system_prompt — 自定义指导todo列表使用的提示词

不提供则使用内置提示词，通常不必提供。

2. tool_description — 自定义write_tools工具的描述信息

不提供则使用内置描述，通常不必提供。

### 2.4.2 案例设计

我们设计一个较为复杂的任务：
1. 任务目标:扫描工作目录，测试并修复工作区下的my_add.py文件.
2. 工具列表:为了实现上述任务，提供一系列工具。
    * list_files：扫描工作目录，列出其中的所有文件
    * read_file：扫描指定文件，返回文件内容
    * write_file：向指定文件写入内容
    * run_tests：运行测试，底层基于pytest实现

分析:
1. pytest会扫描目录下所有以 test_ 开头或以 _test 结尾的文件，视为测试文件
2. 然后执行测试文件中所有以 test 开头的函数
3. 执行出错会打印到控制台，如上所示
4. 测试函数的逻辑是调用my_add.py中的add函数，得不到符合预期的结果则抛出异常。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-terra",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [6]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo_workspace")

@tool
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"

    if not target.exists():
        return f"错误：目录不存在: {path}"

    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)

@tool
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"


In [ ]:
from langchain.agents.middleware import TodoListMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file,run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt="你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
)


response = agent.invoke({
    "messages" : [HumanMessage("请测试并修复工作区下的my_add.py文件中的代码")]
})


rprint(response)# response最后附带 todo字段

{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下的my_add.py文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='bed5aeb5-6755-4559-9c9a-e18192267c7d'
        ),
        AIMessage(
            content=' ',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 94,
                    'prompt_tokens': 5868,
                    'total_tokens': 5962,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 18,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.6-terra',
                'system_fingerprint': None,
                'id': 'resp_ft_b0a01e2cba6b459ca09ddf7adb71f2a7',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10d4d-4ac6-7e62-ae44-ca23059c7134-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '读取并检查工作区及 my_add.py 的现有代码', 'status': 'in_progress'},
                            {'content': '运行测试，定位并修复 my_add.py 中的问题', 'status': 'pending'},
                            {'content': '再次运行测试并确认修复结果', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_fd56add9ff3a4063af6935e0969d5949',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 5868,
                'output_tokens': 94,
                'total_tokens': 5962,
                'input_token_details': {},
                'output_token_details': {'reasoning': 18}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '读取并检查工作区及 my_add.py 的现有代码', 'status': 
'in_progress'}, {'content': '运行测试，定位并修复 my_add.py 中的问题', 'status': 'pending'}, {'content': 
'再次运行测试并确认修复结果', 'status': 'pending'}]",
            name='write_todos',
            id='01854bc1-dee3-429c-bc85-a62f47e3c9c1',
            tool_call_id='call_fd56add9ff3a4063af6935e0969d5949'
        ),
        AIMessage(
            content='我先检查工作区文件和目标模块，确认现有实现及测试入口，然后运行测试获取具体失败信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 82,
                    'prompt_tokens': 6034,
                    'total_tokens': 6116,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.6-terra',
                'system_fingerprint': None,
                'id': 'resp_054ba0a0e76d982f016ac3eadd759487d0b77ef24493163fb9',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10d4d-6471-7860-a1b3-a912927f95f1-0',
            tool_calls=[
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_cwMZm699twVfZdlAoHUMj2CL',
                    'type': 'tool_call'
                },
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_hYU3EvJmDb5jvC0OPauQXWWN',
                    'type': 'tool_call'
                }
            ],
   

### 2.4.3 内部原理

先说结论：**TodoListMiddleware 本身不会做规划，规划是模型做的。** 中间件只是给模型准备好条件：在 state 里加一个存放计划的 `todos` 字段，给模型一个 `write_todos` 工具和一段使用说明，再拦住一种会出错的用法。它由四部分组成（源码在 `.venv/lib/python3.11/site-packages/langchain/agents/middleware/todo.py`，行号对应 langchain 1.4.2）：

| 组成部分 | 源码位置 | 作用 |
|---|---|---|
| `state_schema = PlanningState` | 第 35–42 行、第 201 行 | 给 Agent 的 state 增加 `todos` 字段 |
| `self.tools = [write_todos]` | 第 153 行、第 220 行 | 自动注册 `write_todos` 工具；工具返回 `Command(update=...)`，把计划写进 `todos` |
| `wrap_model_call` 钩子 | 第 231 行 | 每次调用模型前，把 `write_todos` 的使用说明追加到系统提示词后面 |
| `after_model` 钩子 | 第 286 行 | 模型一次发出多个 `write_todos` 调用时，拒绝执行并返回错误 |

#### 1. 给 state 增加 `todos` 字段

Agent 运行过程中的数据都存放在 state 里，可以把它理解成一个字典，默认的主要字段是 `messages`。中间件可以通过 `state_schema` 给 state 增加字段，TodoListMiddleware 增加的是 `todos`：

```python
class Todo(TypedDict):
    content: str                                            # 任务内容
    status: Literal["pending", "in_progress", "completed"]  # 待办 / 进行中 / 已完成

class PlanningState(AgentState):
    todos: Annotated[NotRequired[list[Todo]], OmitFromInput]
```

* `NotRequired`：这个字段可以不存在。模型第一次调用 `write_todos` 之前，state 里没有 `todos`。
* `OmitFromInput`：不能通过 `invoke` 的输入传入，只会出现在输出里。所以 2.4.2 的 `response` 里多了一个顶层的 `'todos'` 字段。

#### 2. `write_todos` 工具：用 Command 写入 state

中间件把 `write_todos` 放在自己的 `tools` 属性里。`create_agent` 建图时，会把所有中间件的 `tools` 和我们传入的 `tools` 合并后交给模型（`langchain/agents/factory.py` 第 1037、1084 行），所以 2.4.2 的 `tools=[...]` 里没有写它，模型也能调用。

工具的实现只有几行（第 153 行，简化）：

```python
def _write_todos(runtime, todos):
    return Command(update={
        "todos": todos,  # 用新列表整体替换 state 中的 todos
        "messages": [ToolMessage(f"Updated todo list to {todos}", tool_call_id=runtime.tool_call_id)],
    })
```

* 这就是 2.2.4 中提到的“工具返回 `Command(update=...)`，直接修改 Agent 的 state”。普通工具只能返回一段文本，由框架包装成 ToolMessage；`write_todos` 返回 `Command`，一次更新两个字段：`todos` 存放计划，`messages` 里追加一条 ToolMessage（每个工具调用都必须有对应的结果消息，就是 2.4.2 输出中的 `"Updated todo list to [...]"`）。
* `runtime` 参数由框架自动传入，用来取得本次调用的 `tool_call_id`，模型看不到它；模型调用时只需要提供 `todos`。

**state 的每个字段有自己的更新规则。** `messages` 字段在定义时带有合并规则 `add_messages`（`AgentState` 中写作 `Annotated[list[AnyMessage], add_messages]`），新消息会追加到列表末尾；`todos` 没有合并规则，新值直接覆盖旧值。所以模型每次调用 `write_todos` 都必须传入完整的列表。对照 2.4.2 的输出：模型调用了 4 次 `write_todos`，每次都带着全部 3 项任务，变化的只是各项的 `status`。

#### 3. `wrap_model_call`：每次调用模型前追加使用说明

`wrap_model_call` 不是图中单独的节点，而是包在 `model` 节点内部，每次调用模型时执行（钩子表见 01 笔记 1.4）。TodoListMiddleware 用它把内置提示词 `WRITE_TODOS_SYSTEM_PROMPT` 追加到系统提示词后面，再把修改后的请求交给模型（第 231–256 行）。它只修改本次请求，不会写入 state。

以 2.4.2 为例，模型实际收到的系统提示词，是 `create_agent` 中的 `system_prompt`（“你是一个代码修复助手……”）后面再接一段英文说明（以 “## `write_todos`” 开头，介绍什么时候使用该工具、如何更新任务状态等）。

需要注意两点：

* **中间件没有把当前的 todos 写进提示词。** 模型之所以“记得”计划，是因为之前每次调用 `write_todos` 的参数和返回的 ToolMessage 都留在 `messages` 里，下次调用模型时会一起发过去。state 中的 `todos` 字段主要给程序使用，例如在前端展示进度。
* **中间件不会强制模型列计划。** 是否调用 `write_todos` 由模型根据提示词自己决定；内置的工具说明明确写着，少于 3 步的简单任务不要使用。

#### 4. `after_model`：不允许一次调用多个 `write_todos`

模型可以在一条 AIMessage 里同时发出多个工具调用（并行调用）。如果其中有两个 `write_todos`，它们会在同一步里同时覆盖 `todos`，引擎无法决定保留哪一个，会直接报错（`InvalidUpdateError`，下面的示例 2 会演示）。

因此 `after_model` 在工具执行之前检查最后一条 AIMessage（第 286 行起）：如果其中的 `write_todos` 调用超过一个，就为每个 `write_todos` 调用生成一条 `status="error"` 的 ToolMessage，内容大意是“不要并行调用 `write_todos`，每次只调用一次”。这些调用已经有了结果消息，就不会再被执行；同一条消息中的其他工具调用照常执行。模型在下一轮看到错误后，会改为只调用一次。这和 HITL 拒绝工具调用的做法相同。

#### 5. 一次完整的执行过程

加入 TodoListMiddleware 后，Agent 的图是：

```
__start__ → model（内含 wrap_model_call）→ TodoListMiddleware.after_model ─┬→ tools → 回到 model
                                                                          └→ __end__（模型没有调用工具时）
```

对照 2.4.2 的运行结果：

1. **model 节点**：`wrap_model_call` 追加使用说明后调用模型，模型返回一个 `write_todos` 调用，列出 3 项任务，第 1 项为 `in_progress`；
2. **after_model 节点**：只有一个 `write_todos`，放行；
3. **tools 节点**：执行 `write_todos`，返回的 `Command` 把 `todos` 写成这 3 项，并追加一条 ToolMessage；
4. **回到 model 节点**：模型在消息历史中看到计划，开始调用 `list_files`、`read_file` 等工具；每完成一步，就再调用一次 `write_todos`，传入更新了状态的完整列表；
5. 最后一次调用模型时，模型不再调用工具，直接给出总结，`after_model` 放行，Agent 结束。`response["todos"]` 就是最后一次写入的列表，3 项都是 `completed`。

#### 6. 动手验证

下面用假模型 `GenericFakeChatModel` 搭一个小 Agent（不调用真实模型，每次输出一致），验证上面的说法：

* 示例 1：打印图中的节点、`invoke` 输入输出包含的字段、模型能调用的工具、模型收到的系统提示词，并用 `stream_mode="updates"` 观察每个节点修改了 state 的哪些字段；
* 示例 2：让模型一次发出两个 `write_todos` 调用，对比有无 `after_model` 检查时的结果。

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool


class FakeToolModel(GenericFakeChatModel):
    """假模型：按顺序返回预先写好的消息，不调用任何接口。"""

    def bind_tools(self, tools, **kwargs):
        return self  # create_agent 会调用 bind_tools，假模型直接返回自身


class TracedFakeModel(FakeToolModel):
    """在假模型的基础上，打印它收到了哪些工具和消息。"""

    def bind_tools(self, tools, **kwargs):
        if not printed_tools:  # 每次调用模型前都会绑定工具，这里只打印一次
            printed_tools.append(True)
            print("模型能调用的工具:", [t.name for t in tools])
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        system = messages[0].text if isinstance(messages[0], SystemMessage) else ""
        print(f"  [model] 收到 {len(messages)} 条消息，系统提示词开头: {system[:40]!r}")
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)


printed_tools = []


@tool
def demo_note(text: str) -> str:
    """记录一条笔记（本例中模型不会调用它）"""
    return text


todo_demo_model = TracedFakeModel(messages=iter([
    AIMessage("", tool_calls=[{"name": "write_todos", "id": "call_1", "args": {"todos": [
        {"content": "读取文件", "status": "in_progress"},
        {"content": "修复并测试", "status": "pending"},
    ]}}]),
    AIMessage("", tool_calls=[{"name": "write_todos", "id": "call_2", "args": {"todos": [
        {"content": "读取文件", "status": "completed"},
        {"content": "修复并测试", "status": "in_progress"},
    ]}}]),
    AIMessage("两步都已完成"),
]))
todo_demo_agent = create_agent(model=todo_demo_model, tools=[demo_note],
                               middleware=[TodoListMiddleware()], system_prompt="你是一个任务助手。")

print("图中的节点:", list(todo_demo_agent.get_graph().nodes))
print("invoke 输入允许的字段:", list(todo_demo_agent.get_input_jsonschema()["properties"]))
print("invoke 输出包含的字段:", list(todo_demo_agent.get_output_jsonschema()["properties"]))
print()

for chunk in todo_demo_agent.stream({"messages": [HumanMessage("帮我完成两步任务")]}, stream_mode="updates"):
    for node, update in chunk.items():
        if not update:
            print(f"  [{node}] 没有修改 state")
            continue
        line = f"  [{node}] 修改了 state 的字段: {list(update)}"
        if "todos" in update:
            line += f"，todos = {[(t['content'], t['status']) for t in update['todos']]}"
        print(line)

图中的节点: ['__start__', 'model', 'tools', 'TodoListMiddleware.after_model', '__end__']
invoke 输入允许的字段: ['messages']
invoke 输出包含的字段: ['messages', 'structured_response', 'todos']

模型能调用的工具: ['write_todos', 'demo_note']
  [model] 收到 2 条消息，系统提示词开头: '你是一个任务助手。\n\n## `write_todos`\n\nYou have ac'
  [model] 修改了 state 的字段: ['messages']
  [TodoListMiddleware.after_model] 没有修改 state
  [tools] 修改了 state 的字段: ['todos', 'messages']，todos = [('读取文件', 'in_progress'), ('修复并测试', 'pending')]
  [model] 收到 4 条消息，系统提示词开头: '你是一个任务助手。\n\n## `write_todos`\n\nYou have ac'
  [model] 修改了 state 的字段: ['messages']
  [TodoListMiddleware.after_model] 没有修改 state
  [tools] 修改了 state 的字段: ['todos', 'messages']，todos = [('读取文件', 'completed'), ('修复并测试', 'in_progress')]
  [model] 收到 6 条消息，系统提示词开头: '你是一个任务助手。\n\n## `write_todos`\n\nYou have ac'
  [model] 修改了 state 的字段: ['messages']
  [TodoListMiddleware.after_model] 没有修改 state


对照示例 1 的输出：

* 图中只有 `TodoListMiddleware.after_model` 一个中间件节点，`wrap_model_call` 没有单独的节点（第 3 部分）。
* `invoke` 的输入只允许 `messages`，输出多了 `todos`（第 1 部分）。
* 模型能调用的工具里有 `write_todos`，虽然 `tools=[...]` 中没有写它（第 2 部分）。
* 模型每次收到的系统提示词都是“你是一个任务助手。”后面接着 “## `write_todos`” 说明；消息条数从 2 条增加到 4 条、6 条，之前的 `write_todos` 调用和 ToolMessage 都在其中（第 3 部分）。
* 修改 `todos` 的是 `tools` 节点，也就是 `write_todos` 返回的 `Command`，它同时修改了 `todos` 和 `messages`；第二次写入时，`todos` 被整体替换成新列表（第 2 部分）。

示例 2：

In [ ]:
@tool
def demo_step(x: str) -> str:
    """执行一步操作"""
    print(f"    >>> demo_step 真的执行了，x={x}")
    return f"完成 {x}"


# 模型在一条 AIMessage 里同时发出两个 write_todos 调用和一个普通工具调用
todo_parallel_call = AIMessage("", tool_calls=[
    {"name": "write_todos", "id": "call_w1", "args": {"todos": [{"content": "计划A", "status": "in_progress"}]}},
    {"name": "write_todos", "id": "call_w2", "args": {"todos": [{"content": "计划B", "status": "pending"}]}},
    {"name": "demo_step", "id": "call_s1", "args": {"x": "1"}},
])

print("① 使用 TodoListMiddleware：")
todo_agent_ok = create_agent(model=FakeToolModel(messages=iter([todo_parallel_call, AIMessage("好的")])),
                             tools=[demo_step], middleware=[TodoListMiddleware()])
todo_result_ok = todo_agent_ok.invoke({"messages": [HumanMessage("开始")]})
for msg in todo_result_ok["messages"]:
    if msg.type == "tool":
        print(f"  ToolMessage[{msg.tool_call_id}] status={msg.status}: {msg.content[:60]}")
print("  todos:", todo_result_ok.get("todos"))


class TodoWithoutCheck(TodoListMiddleware):
    def after_model(self, state, runtime):
        return None  # 去掉并行调用检查


print("\n② 去掉 after_model 的检查：")
todo_agent_bad = create_agent(model=FakeToolModel(messages=iter([todo_parallel_call, AIMessage("好的")])),
                              tools=[demo_step], middleware=[TodoWithoutCheck()])
try:
    todo_agent_bad.invoke({"messages": [HumanMessage("开始")]})
except Exception as e:
    print(f"  {type(e).__name__}: {str(e).splitlines()[0]}")

① 使用 TodoListMiddleware：
    >>> demo_step 真的执行了，x=1
  ToolMessage[call_w1] status=error: Error: The `write_todos` tool should never be called multipl
  ToolMessage[call_w2] status=error: Error: The `write_todos` tool should never be called multipl
  ToolMessage[call_s1] status=success: 完成 1
  todos: None

② 去掉 after_model 的检查：
    >>> demo_step 真的执行了，x=1
  InvalidUpdateError: At key 'todos': Can receive only one value per step. Use an Annotated key to handle multiple values.


对照示例 2 的输出：

* 有 `after_model` 检查时：两个 `write_todos` 调用都收到了 `status=error` 的 ToolMessage，没有执行，所以 `todos` 仍然不存在；同一条消息中的 `demo_step` 照常执行。
* 去掉检查后：两个 `write_todos` 都被执行，在同一步里同时覆盖 `todos`，引擎直接抛出 `InvalidUpdateError`，整个运行中断。这就是 `after_model` 要做这项检查的原因。